This file is basically a way to covert our pose -> 3d Keypoints -> SMPL-X mesh 

The idea is that this would be blended with our front view point cloud to generate the sides and back estimations of our final 3D model

In [1]:
import cv2
import numpy as np
import torch
import trimesh
import json
import os
import subprocess
import matplotlib.pyplot as plt
from smplx import SMPLX
from human_body_prior.tools.model_loader import load_vposer
import plotly.graph_objects as go
from tqdm import trange
import yaml

In [2]:
# Load configurations
with open("/Users/adeleyounis/Desktop/Capstone/wAI/config.yaml", "r") as f:
    config = yaml.safe_load(f)
    
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback if running in notebook or REPL
    BASE_DIR = os.getcwd()

def resolve_path(rel_path):
    return os.path.join(BASE_DIR, rel_path)

# Resolve all paths to search within repo
paths = {k: resolve_path(v) for k, v in config["paths"].items()}

In [3]:
# load SMPL-X model in

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

smplx_model_path = '/Users/adeleyounis/Desktop/Capstone/wAI/volume-model/models/smplx'
vpose_model_path = '/Users/adeleyounis/Desktop/Capstone/wAI/volume-model/models/vposer_v1_0'
batch_size = 1

vp_model, pose_prior = load_vposer(paths["vposer_model_path"], vp_model='snapshot')
vp_model = vp_model.to(device)
vp_model.eval()
print("VPoser loaded successfully!")

# TODO: adjust gender by reading in img csv file
# load in SMPL-X model
smplx_model = SMPLX(model_path=paths["smplx_model_path"], 
                    gender='FEMALE', # options are MALE, FEMALE, NEUTRAL
                    use_pca=True,      # PCA for hands - simplifies hand pose representation if True
                    batch_size=batch_size) 

Using device: cpu
Found Trained Model: /Users/adeleyounis/Desktop/Capstone/wAI/volume-model/models/vposer_v1_0/snapshots/TR00_E096.pt
VPoser loaded successfully!


In [4]:
# load in RGB and depth images
rgb_path = paths["rgb_img_path"]
depth_path = paths["depth_img_path"]

rgb_image = cv2.imread(rgb_path)

# support for .npy depth images and png/jpg depth images for depth images
if depth_path.endswith('.npy'):
    depth_image = np.load(depth_path)
else:  
    depth_image = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)

H, W = rgb_image.shape[:2]

print("RGB image shape:", rgb_image.shape)
print("Depth image shape:", depth_image.shape)

RGB image shape: (480, 640, 3)
Depth image shape: (480, 640)


In [7]:
# run OpenPose to get 3D keypoints from RGB image
pose_dir = paths["pose_output_dir"]

cmd = [
    "python3",
    paths["openpose_path"],
    "--input", rgb_path,
    "--write_json", pose_dir    
]

result = subprocess.run(cmd, capture_output=True, text=True) 

In [6]:
keypoints_json = pose_dir = paths["pose_output_dir"] + "pose_keypoints.json"

with open(keypoints_json) as f:
    data = json.load(f)

keypoints_2d = np.array(data["people"][0]["pose_keypoints_2d"]).reshape(-1, 2)
print("Loaded", keypoints_2d.shape[0], "keypoints") # 19 points

FileNotFoundError: [Errno 2] No such file or directory: '/Users/adeleyounis/Desktop/Capstone/wAI/volume-model/output/pose_keypoints.json'

In [ ]:
# map openpose to smplx keypoints
# 18 joints to 22 joints
openpose18_to_smplx = {
    8: 0,    # RHip -> Pelvis (use average of LHip & RHip ideally)
    11: 1,   # LHip -> LHip
    8: 2,    # RHip -> RHip
    1: 3,    # Neck -> Spine1
    12: 4,   # LKnee -> LKnee
    9: 5,    # RKnee -> RKnee
    13: 7,   # LAnkle -> LAnkle
    10: 8,   # RAnkle -> RAnkle
    5: 16,   # LShoulder -> LShoulder
    2: 17,   # RShoulder -> RShoulder
    6: 18,   # LElbow -> LElbow
    3: 19,   # RElbow -> RElbow
    7: 20,   # LWrist -> LWrist
    4: 21,   # RWrist -> RWrist
    0: 15,   # Nose -> Head
}

keypoints_2d_openpose = np.array(keypoints_2d)  # shape (18, 2)
keypoints_2d_smplx = np.zeros((22, 2))    # initialize SMPL-X order

# map pelvis
lhip, rhip = keypoints_2d[11, :2], keypoints_2d[8, :2]
keypoints_2d_smplx[0] = (lhip + rhip) / 2


for openpose_id, smplx_id in openpose18_to_smplx.items():
    if openpose_id < len(keypoints_2d):
        keypoints_2d_smplx[smplx_id] = keypoints_2d[openpose_id, :2]

print("Mapped 2D keypoints to SMPL-X format:", keypoints_2d_smplx.shape) # want 22
print("Valid keypoints:", np.sum(~np.isnan(keypoints_2d_smplx[:, 0]))) # check none are empty (want 22 again)

In [ ]:
# load depth values for each 2D keypoint
z_vals = []
for (x, y) in keypoints_2d.astype(int):
    if 0 <= x < depth_image.shape[1] and 0 <= y < depth_image.shape[0]:
        print(f"({x}, {y}) -> depth = {depth_image[y, x]}")
        z = depth_image[y, x]
        if z > 0:
            z_vals.append(z)
        else:
            z_vals.append(0)
    else:
        z_vals.append(0)

z_vals = np.array(z_vals)
keypoints_3d = np.zeros((len(keypoints_2d), 3))

for i, ((x, y), z) in enumerate(zip(keypoints_2d, z_vals)):
    if z > 0:
        X = (x - config["camera"]["cx"]) * z / config["camera"]["fx"]
        Y = (y - config["camera"]["cy"]) * z / config["camera"]["fy"]
        Z = z
        keypoints_3d[i] = [X, Y, Z]
    else:
        keypoints_3d[i] = [np.nan, np.nan, np.nan]


In [ ]:
POSE_PAIRS = [ ["Neck", "RShoulder"], ["Neck", "LShoulder"], ["RShoulder", "RElbow"],
               ["RElbow", "RWrist"], ["LShoulder", "LElbow"], ["LElbow", "LWrist"],
               ["Neck", "RHip"], ["RHip", "RKnee"], ["RKnee", "RAnkle"], ["Neck", "LHip"],
               ["LHip", "LKnee"], ["LKnee", "LAnkle"], ["Neck", "Nose"], ["Nose", "REye"],
               ["REye", "REar"], ["Nose", "LEye"], ["LEye", "LEar"] ]

# Map body part names to indices (depends on your keypoints)
BODY_PARTS = { "Nose": 0, "Neck": 1, "RShoulder": 2, "RElbow": 3, "RWrist": 4,
               "LShoulder": 5, "LElbow": 6, "LWrist": 7, "RHip": 8, "RKnee": 9,
               "RAnkle": 10, "LHip": 11, "LKnee": 12, "LAnkle": 13, "REye": 14,
               "LEye": 15, "REar": 16, "LEar": 17, "Background": 18 }

head_indices = [
    BODY_PARTS["REye"],
    BODY_PARTS["LEye"],
    BODY_PARTS["REar"],
    BODY_PARTS["LEar"]
]

In [ ]:
# custom pose augs
keypoints_3d[BODY_PARTS["RKnee"], 2] = keypoints_3d[BODY_PARTS["LKnee"], 2]
keypoints_3d[BODY_PARTS["RAnkle"], 1] = keypoints_3d[BODY_PARTS["LAnkle"], 1]
keypoints_3d[BODY_PARTS["RAnkle"], 2] = keypoints_3d[BODY_PARTS["LAnkle"], 2]
keypoints_3d[BODY_PARTS["RAnkle"], 0] = keypoints_3d[BODY_PARTS["RKnee"], 0]
keypoints_3d[BODY_PARTS["LWrist"], 2] = keypoints_3d[BODY_PARTS["LElbow"], 2]

In [ ]:
fig = go.Figure()

# --- Skeleton Lines ---
for pair in POSE_PAIRS:
    idFrom = BODY_PARTS[pair[0]]
    idTo = BODY_PARTS[pair[1]]

    if idFrom in head_indices or idTo in head_indices:
        continue

    fig.add_trace(go.Scatter3d(
        x=[keypoints_3d[idFrom, 0], keypoints_3d[idTo, 0]],
        y=[keypoints_3d[idFrom, 1], keypoints_3d[idTo, 1]],
        z=[keypoints_3d[idFrom, 2], keypoints_3d[idTo, 2]],
        mode='lines',
        line=dict(color='blue', width=4),
        showlegend=False
    ))

# --- Add Labels for Each Joint ---
for part, idx in BODY_PARTS.items():
    if idx == BODY_PARTS["Background"]:  # skip background
        continue
    if idx in head_indices:  # optional: skip head
        continue

    x, y, z = keypoints_3d[idx]
    fig.add_trace(go.Scatter3d(
        x=[x], y=[y], z=[z],
        mode='text',
        text=[part],
        textposition='top center',
        textfont=dict(size=10, color='black'),
        showlegend=False
    ))

# --- Clean Display (No Axes or Grids) ---
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode='data'
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(l=0, r=0, b=0, t=0),
)

fig.show()


In [ ]:
keypoints_3d_torch = torch.tensor(keypoints_3d, dtype=torch.float32).unsqueeze(0)

# --- Step 3: Define parameters to optimize ---
body_pose = torch.zeros([1, 63], dtype=torch.float32, requires_grad=True)  # 21x3
global_orient = torch.zeros([1, 3], dtype=torch.float32, requires_grad=True)
betas = torch.zeros([1, smplx_model.num_betas], dtype=torch.float32, requires_grad=True)
transl = torch.zeros([1, 3], dtype=torch.float32, requires_grad=True)

# --- Step 4: Valid joint mask (exclude head/face) ---
valid_mask = torch.ones(keypoints_3d_torch.shape[1], dtype=torch.bool)
head_indices = [0, 14, 15, 16, 17]  # Adjust per dataset
valid_mask[head_indices] = False


def keypoint_loss_fn(joints_3d, keypoints_3d):
    mask = valid_mask.unsqueeze(0).unsqueeze(2)
    diff = (joints_3d - keypoints_3d) * mask
    return (diff ** 2).sum() / mask.sum()

smplx_indices = list(range(keypoints_3d_torch.shape[1]))

# --- Step 6: Stage 1 — optimize global orientation & translation ---
optimizer = torch.optim.Adam([global_orient, transl], lr=0.005)
for i in trange(500, desc="Stage 1: Orient/Transl"):
    output = smplx_model(
        body_pose=body_pose,
        global_orient=global_orient,
        betas=betas,
        transl=transl
    )
    joints_3d = output.joints[:, smplx_indices, :]
    loss = keypoint_loss_fn(joints_3d, keypoints_3d_torch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 50 == 0:
        print(f"[Stage 1] Iter {i}, loss={loss.item():.5f}")

# --- Stage 2: optimize body pose ---
optimizer = torch.optim.Adam([body_pose], lr=0.005)
for i in trange(500, desc="Stage 2: Body Pose"):
    output = smplx_model(
        body_pose=body_pose,
        global_orient=global_orient,
        betas=betas,
        transl=transl
    )
    joints_3d = output.joints[:, smplx_indices, :]
    loss = keypoint_loss_fn(joints_3d, keypoints_3d_torch)
    loss += 0.001 * torch.norm(body_pose)  # regularization

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 50 == 0:
        print(f"[Stage 2] Iter {i}, loss={loss.item():.5f}")

# --- Stage 3: optimize body shape (betas) ---
optimizer = torch.optim.Adam([betas], lr=0.005)
for i in trange(500, desc="Stage 3: Shape"):
    output = smplx_model(
        body_pose=body_pose,
        global_orient=global_orient,
        betas=betas,
        transl=transl
    )
    joints_3d = output.joints[:, smplx_indices, :]
    loss = keypoint_loss_fn(joints_3d, keypoints_3d_torch)
    loss += 0.001 * torch.norm(betas)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 50 == 0:
        print(f"[Stage 3] Iter {i}, loss={loss.item():.5f}")

print("\nFinal losses:")
print(f"  Pose loss: {loss.item():.5f}")
print("  Translation:", transl.detach().numpy())
print("  Global orient (rad):", global_orient.detach().numpy())

import matplotlib.pyplot as plt
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*keypoints_3d.T, color='r', label='Target 3D')
ax.scatter(*output.joints[0, smplx_indices, :].detach().cpu().numpy().T, color='b', label='SMPL-X Fit')
ax.legend()
plt.show()

In [ ]:
vertices = output.vertices[0].detach().cpu().numpy()
faces = smplx_model.faces

mesh = trimesh.Trimesh(vertices, faces)
mesh.show()

In [ ]:
voxel_size = config["voxelization"]["voxel_size"] 
voxel_grid = mesh.voxelized(pitch=voxel_size)

# convert to dense occupancy grid (numpy array)
voxels_dense = voxel_grid.matrix  # 3D boolean array
print("Voxel grid shape:", voxels_dense.shape)
print("Voxel size (m):", voxel_size)

# visualize voxel grid
voxel_grid.show()